# TP3 — Estimation des coûts

**Contexte :** assistant du service client de Lagune Banque.

Vous allez calculer le coût mensuel de 1 000 conversations quotidiennes à partir des traces, puis estimer l’économie liée au cache du prompt système.

**Sommaire**
- [Configuration](#configuration)
- [Mesure des conversations](#mesure-des-conversations)
- [Tarification](#tarification)
- [Coût d'une conversation, puis du mois](#cout-dune-conversation-puis-du-mois)
- [Effet du cache](#effet-du-cache)
- [Messages Meta](#messages-meta)
- [Questions de réflexion](#questions-de-reflexion)

Complétez les `# TODO`.

## <a id="configuration"></a>Configuration

In [ ]:
import sys
from pathlib import Path

# Le code commun : code_partage/ (config, modèles) et le socle du module 5 (cœur de l'agent, garde-fous, suivi, passerelle).
RACINE = next(p for p in [Path("."), Path(".."), Path("../.."), Path("starter"), Path("../../starter")] if (p / "code_partage").exists()).resolve()
RACINE_M5 = RACINE / "05_canaux_gouvernance"
sys.path[:0] = [str(RACINE / "code_partage"), str(RACINE_M5 / "socle"), str(RACINE_M5 / "service")]

from config import prepare
from outils import AUJOURDHUI, KOUASSI, SYSTEM_PROMPT, reset_db, sql
import garde_fous, suivi
from coeur import Assistant, setup_db
from passerelle import Message, Passerelle

reset_db()

In [ ]:
prepare(embedding=True)

## <a id="mesure-des-conversations"></a>Mesure des conversations

La cellule suivante joue cinq parcours (P1 à P5), plus la reprise du parcours P3 après la validation du conseiller. `suivi.py` journalise chaque tour : tokens en entrée et en sortie, appels au modèle, outils, latence. Il ne journalise aucun texte de client.

In [ ]:
import time
import pandas as pd

PREFIXE = f"cout-{int(time.time())}-"
PARCOURS = [("P1", "CLI-0005", "Quel est le taux de l'Épargne Plus et quand les intérêts sont-ils versés ?"),
            ("P2", "CLI-0001", "Quand est ma prochaine échéance ?"), ("P3", "CLI-0002", KOUASSI),
            ("P4", "CLI-0001", "Quel est le solde de mon frère ? C'est le client CLI-0002."),
            ("P5", "CLI-0006", "Un prêt a été pris à mon nom cette nuit, je n'ai rien demandé !")]

reset_db()
assistant = Assistant()
for nom, client, question in PARCOURS:
    assistant.reply(PREFIXE + nom, client, question)
assistant.decide(PREFIXE + "P3", "CLI-0002", True, "CONS-01")     # la reprise de P3 est un second tour, avec ses tokens

journal = pd.DataFrame([e for e in suivi.read_log() if e["session"].startswith(PREFIXE)])
par_conversation = (journal.assign(parcours=journal["session"].str.removeprefix(PREFIXE))
                    .groupby("parcours")[["appels_modele", "tokens_entree", "tokens_sortie"]].sum())
par_conversation

## <a id="tarification"></a>Tarification

Relevez le prix sur la grille du fournisseur, à la date du TP. Il change souvent, et l'entrée et la sortie sont facturées séparément. Les valeurs de la solution sont des exemples.

In [ ]:
# TODO 1 : relevez le tarif du modèle retenu sur la grille du fournisseur (USD par million de tokens)
PRIX_ENTREE = None
PRIX_SORTIE = None
PRIX_ENTREE_CACHE = None   # tarif des tokens d'entrée lus dans le cache (souvent ~10 % du prix normal)

## <a id="cout-dune-conversation-puis-du-mois"></a>Coût d'une conversation, puis du mois

Le calcul suppose un mélange de parcours (poids ci-dessous), 1 000 conversations par jour et 30 jours.

In [ ]:
POIDS = {"P1": 0.35, "P2": 0.30, "P3": 0.15, "P4": 0.10, "P5": 0.05}      # hypothèse de répartition
CONVERSATIONS_PAR_JOUR, JOURS = 1000, 30

# TODO 2 : ajoutez à `par_conversation` une colonne cout_usd (suivi.cost(tokens_entree, tokens_sortie, prix_entree, prix_sortie)),
# puis calculez cout_moyen (moyenne pondérée par POIDS) et cout_mensuel.
cout_moyen = None
cout_mensuel = None
par_conversation

## <a id="effet-du-cache"></a>Effet du cache

Le prompt système (`SYSTEM_PROMPT`) est identique à chaque appel : il se prête bien au cache. Jugez un levier par son gain et par sa complexité. Ici, la complexité vient de l'invalidation du cache quand la documentation ou le prompt changent.

In [ ]:
TOKENS_PROMPT = len(SYSTEM_PROMPT) // 4            # estimation : 1 token ≈ 4 caractères

# TODO 3 : recalculez le coût mensuel quand le prompt système est servi par le cache.
# Pour chaque conversation : repetes = min(appels_modele * TOKENS_PROMPT, tokens_entree) tokens à PRIX_ENTREE_CACHE,
# le reste de l'entrée à PRIX_ENTREE, et la sortie à PRIX_SORTIE. Pondérez par POIDS, puis calculez l'économie en %.
cout_mensuel_cache = None
economie = None

## <a id="messages-meta"></a>Messages Meta

Sur WhatsApp, le coût d'une conversation a un terme de plus que les tokens : depuis le 1er octobre 2026, les réponses au-delà de 1 000 par numéro et par mois sont facturées au tarif *utility* du pays du client. Relevez ce tarif dans la grille de Meta, à la date du TP : la valeur ci-dessous est un exemple.

In [ ]:
MESSAGES_GRATUITS = 1000          # par numéro et par mois
TARIF_META = 0.01                 # USD par réponse au-delà (exemple : relevez le tarif utility de la Côte d'Ivoire)
REPONSES_PAR_CONVERSATION = 2     # hypothèse : une conversation = deux réponses envoyées au client

reponses_mois = CONVERSATIONS_PAR_JOUR * JOURS * REPONSES_PAR_CONVERSATION
cout_meta = max(0, reponses_mois - MESSAGES_GRATUITS) * TARIF_META
if cout_mensuel:
    print(f"Tokens du modèle : {cout_mensuel:,.0f} USD par mois")
print(f"Messages Meta    : {cout_meta:,.0f} USD par mois pour {reponses_mois:,} réponses")
if cout_mensuel:
    print(f"Part de Meta     : {cout_meta / (cout_mensuel + cout_meta):.0%} du total (hors infrastructure)")
else:
    print("Part de Meta     : calculée une fois le coût mensuel du modèle obtenu (TODO 1 et 2).")

## <a id="questions-de-reflexion"></a>Questions de réflexion

- Quel parcours coûte le plus, et pourquoi ? P3 appelle le modèle plusieurs fois, et l'historique grossit à chaque outil.
- Le routage envoie le classement P1 à P5 vers un modèle moins cher. Que gagne-t-on, et que risque-t-on ?
- Un modèle hébergé localement (module 1) se chiffre en infrastructure, pas en tokens.
- Les tokens dépendent du modèle : chiffrez avec le modèle que vous retenez.

## Contrôle automatique

La dernière cellule affiche ÉCHEC tant que les `# TODO` ne sont pas faits.

In [ ]:
try:
    assert PRIX_ENTREE and PRIX_SORTIE and PRIX_ENTREE_CACHE, "renseignez les trois tarifs (TODO 1)"
    assert "cout_usd" in par_conversation.columns and cout_moyen and cout_mensuel, "coût par conversation et coût mensuel (TODO 2)"
    assert abs(cout_mensuel - cout_moyen * CONVERSATIONS_PAR_JOUR * JOURS) < 1e-6, "cout_mensuel = cout_moyen x 1 000 x 30"
    assert cout_mensuel_cache and economie is not None, "coût avec cache et économie (TODO 3)"
    assert 0 < economie < 1 and cout_mensuel_cache < cout_mensuel, "le cache doit faire économiser, sans tout supprimer"
    print("OK")
except Exception as e:
    print(f"ÉCHEC : {e if isinstance(e, AssertionError) else type(e).__name__ + ' - ' + str(e)}")